In [10]:
import numpy as np

def gini_impurity(y):
    """Calculate the Gini impurity of a label array."""
    if(len(y) == 0):
        return 0
    classes, counts = np.unique(y, return_counts=True)
    probabilities = counts / y.size
    return 1-np.sum(probabilities ** 2)

class Node:
    def __init__(self, feature_index = None, threshold = None, left = None, right = None, value = None):
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

def split_dataset(X,y, feature_index, threshold):
    """Split the dataset based on a feature and threshold."""
    X_left= X[X[:,feature_index]<=threshold]
    X_right = X[X[:,feature_index]>threshold]
    y_left = y[X[:,feature_index]<=threshold]
    y_right = y[X[:,feature_index]>threshold]
    return X_left,y_left,X_right,y_right

def calculate_loss( y_left, y_right,total):
    weight_left = len(y_left) / total
    weight_right = len(y_right) / total

    child_impurity = weight_left * gini_impurity(y_left) + weight_right * gini_impurity(y_right)
    return child_impurity


def _best_split(X, y, max_features):
    best_loss = float('inf')
    best_feature = None
    best_threshold = None

    feature_idx = np.random.choice(range(X.shape[1]),size = max_features,replace=False)

    for j in feature_idx:
        feature_column = X[:,j]
        unique_feature_column = np.unique(feature_column)

        for t in unique_feature_column:
            X_left,y_left,X_right,y_right = split_dataset(X,y,j,t)

            if len(y_right)==0 or len(y_left)==0:
                continue

            current_loss_function = calculate_loss(y_left,y_right,len(y))

            if current_loss_function< best_loss:
                best_loss = current_loss_function
                best_feature = j
                best_threshold = t


    return best_feature,best_threshold

In [11]:

def get_leaf_value(y):
    return np.argmax(np.bincount(y))
class DecisionTreeClassifier:
    def __init__(self, min_samples_split=2, max_depth=100, max_features = None):
            self.root = None
            self.min_samples_split = min_samples_split
            self.max_depth = max_depth
            self.max_features = max_features

    def fit(self,X,y):
          if self.max_features is None:
               self.max_features = int(np.sqrt(X.shape[1]))
          self.root = self._grow_tree(X,y)

    def _grow_tree(self, X, y, current_depth = 0):
          if current_depth >= self.max_depth or X.shape[0] < self.min_samples_split or np.unique(y).size == 1:
                leaf_value = get_leaf_value(y)
                return Node(value = leaf_value)

          feature_index,threshold = _best_split(X,y,self.max_features)

          if feature_index is None:
                leaf_value = get_leaf_value(y)
                return Node(value = leaf_value)

          X_left,y_left,X_right,y_right = split_dataset(X,y,feature_index,threshold)
          
          left_child = self._grow_tree(X_left,y_left,current_depth+1)
          right_child = self._grow_tree(X_right,y_right,current_depth+1)


          return Node(feature_index,threshold,left_child,right_child)

    def predict(self, X):
          predictions = []
          for x in X:
            predictions.append(self._traverse_tree(x,self.root))
          return np.array(predictions)

    def _traverse_tree(self, x, node):
        if node.value is not None:
            return node.value
   
        feature_index = node.feature_index
        threshold = node.threshold
   
        if(x[feature_index]<=threshold):
            return self._traverse_tree(x,node.left)
        return self._traverse_tree(x,node.right)
   


In [12]:
def _bootstrap_sampling(X,y):
    row_idx = np.random.choice(range(X.shape[0]),size = X.shape[0],replace = True)
    return X[row_idx],y[row_idx]

In [13]:
class RandomForestClassifier:
    def __init__(self, n_estimators=10, max_depth=10, min_samples_split=2, max_features=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.trees = []

    def fit(self,X,y):
        self.trees = []

        for i in range(self.n_estimators):
            X_sample,y_sample = _bootstrap_sampling(X,y)
            tree = DecisionTreeClassifier(self.min_samples_split,self.max_depth,self.max_features)
            tree.fit(X_sample,y_sample)
            self.trees.append(tree)

    def predict(self,X):
        predictions = []

        for tree in self.trees:
            predictions.append(tree.predict(X))

        return np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=1, arr = np.array(predictions).T)
        

In [14]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier as SklearnRF
from sklearn.metrics import accuracy_score

# 1. Generate a synthetic dataset (binary classification)
X, y = make_classification(n_samples=500, n_features=10, n_informative=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Train and Predict with YOUR Scratch Model
# (Using int(np.sqrt(10)) = 3 for max_features)
scratch_rf = RandomForestClassifier(n_estimators=15, max_depth=10, max_features=3)
scratch_rf.fit(X_train, y_train)
scratch_preds = scratch_rf.predict(X_test)
scratch_acc = accuracy_score(y_test, scratch_preds)

# 3. Train and Predict with Scikit-Learn
sklearn_rf = SklearnRF(n_estimators=15, max_depth=10, max_features=3, random_state=42)
sklearn_rf.fit(X_train, y_train)
sklearn_preds = sklearn_rf.predict(X_test)
sklearn_acc = accuracy_score(y_test, sklearn_preds)

print(f"Scratch RF Accuracy: {scratch_acc:.4f}")
print(f"Sklearn RF Accuracy: {sklearn_acc:.4f}")

Scratch RF Accuracy: 0.9000
Sklearn RF Accuracy: 0.9000
